# Gemini OCR Notebook

This notebook keeps the strong parts of the original Gemini pipeline, removes noisy debug-only branches, and adds optional support for preprocessed table and row crops.

Main changes:
- prefer preprocessed table crops when the dataset is available
- use simple editable path variables at the top
- keep total-based repair, but cut extra reporting clutter
- use row and vote-cell crops to rescue blank or suspicious votes


In [1]:
!pip -q install google-genai rapidfuzz


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.7 MB/s eta 0:00:00a 0:00:01


## Setup


In [2]:
import json
import os
import re
import time
import traceback
from collections import defaultdict
from io import BytesIO
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd
from PIL import Image
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

from google import genai
from google.genai import types

MODEL_NAME = os.getenv("GEMINI_MODEL", "gemini-3.1-flash-lite-preview")
OUTPUT_NAME = os.getenv("OUTPUT_NAME", "submission_gemini_clean_plus.csv")

MAX_RETRIES = 3
BACKOFF_BASE = 2.0
RATE_LIMIT_WAIT = 10
MAX_IMAGE_SIDE = 2200
JPEG_QUALITY = 88

STRICT_MATCH_THRESHOLD = 92
LOOSE_MATCH_THRESHOLD = 78
TOTAL_SOFT_TOLERANCE_ABS = 5
TOTAL_HARD_SKIP_PCT = 0.08
TOTAL_VALIDATION_ROUNDS = 1

PREFER_PREPROCESSED_TABLES = True
ENABLE_CROP_RESCUE = True
SAVE_FAILED_RAW_ONLY = True
FILL_EMPTY_WITH_ZERO = True

RAW_IMAGE_DIR = Path("/kaggle/input/competitions/super-ai-engineer-season-6-ocr-2569/final_data/images")
PREPROCESSED_ROOT = Path("/kaggle/input/datasets/guntinunsawatvong/codex-ocr-preprocess/preprocessed")
TEMPLATE_PATH = Path("/kaggle/input/datasets/guntinunsawatvong/data-thai/data/submission_template.csv")

IMAGES_ROOT = PREPROCESSED_ROOT / "tables" if PREFER_PREPROCESSED_TABLES else RAW_IMAGE_DIR

WORK_DIR = Path("/kaggle/working/gemini_clean_plus")
WORK_DIR.mkdir(parents=True, exist_ok=True)
FAILED_RAW_DIR = WORK_DIR / "failed_raw"
FAILED_RAW_DIR.mkdir(parents=True, exist_ok=True)

DOC_CACHE_PATH = WORK_DIR / "doc_cache.json"
ROW_CACHE_PATH = WORK_DIR / "row_cache.json"
PROGRESS_PATH = WORK_DIR / "progress.json"
OUTPUT_PATH = WORK_DIR / OUTPUT_NAME

ROWS_MANIFEST_PATH = PREPROCESSED_ROOT / "manifests/rows.csv"

print(f"MODEL_NAME      : {MODEL_NAME}")
print(f"IMAGES_ROOT     : {IMAGES_ROOT}")
print(f"TEMPLATE_PATH   : {TEMPLATE_PATH}")
print(f"ROWS_MANIFEST   : {ROWS_MANIFEST_PATH}")
print(f"OUTPUT_PATH     : {OUTPUT_PATH}")


MODEL_NAME      : gemini-3.1-flash-lite-preview
IMAGES_ROOT     : /kaggle/input/datasets/guntinunsawatvong/codex-ocr-preprocess/preprocessed/tables
TEMPLATE_PATH   : /kaggle/input/datasets/guntinunsawatvong/data-thai/data/submission_template.csv
ROWS_MANIFEST   : /kaggle/input/datasets/guntinunsawatvong/codex-ocr-preprocess/preprocessed/manifests/rows.csv
OUTPUT_PATH     : /kaggle/working/gemini_clean_plus/submission_gemini_clean_plus.csv


In [3]:
api_key = os.environ.get("GEMINI_API_KEY", "").strip()
if not api_key:
    raise ValueError("Please set GEMINI_API_KEY in Kaggle secrets or environment.")

client = genai.Client(api_key=api_key)

## Utilities


In [4]:
THAI_DIGIT_MAP = str.maketrans("\u0e50\u0e51\u0e52\u0e53\u0e54\u0e55\u0e56\u0e57\u0e58\u0e59", "0123456789")
THAI_UNITS = {
    "\u0e28\u0e39\u0e19\u0e22\u0e4c": 0,
    "\u0e2b\u0e19\u0e36\u0e48\u0e07": 1,
    "\u0e40\u0e2d\u0e47\u0e14": 1,
    "\u0e2a\u0e2d\u0e07": 2,
    "\u0e2a\u0e32\u0e21": 3,
    "\u0e2a\u0e35\u0e48": 4,
    "\u0e2b\u0e49\u0e32": 5,
    "\u0e2b\u0e01": 6,
    "\u0e40\u0e08\u0e47\u0e14": 7,
    "\u0e41\u0e1b\u0e14": 8,
    "\u0e40\u0e01\u0e49\u0e32": 9,
    "\u0e22\u0e35\u0e48": 2,
}
THAI_MULTIPLIERS = {
    "\u0e2a\u0e34\u0e1a": 10,
    "\u0e23\u0e49\u0e2d\u0e22": 100,
    "\u0e1e\u0e31\u0e19": 1000,
    "\u0e2b\u0e21\u0e37\u0e48\u0e19": 10000,
    "\u0e41\u0e2a\u0e19": 100000,
    "\u0e25\u0e49\u0e32\u0e19": 1000000,
}


def save_json(path: Path, obj: Any) -> None:
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")


def load_json(path: Path, default: Any) -> Any:
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return default


def thai_text_to_number(text: str) -> int:
    text = str(text or "").strip()
    if not text:
        return -1

    vocab = {}
    vocab.update(THAI_UNITS)
    vocab.update(THAI_MULTIPLIERS)

    tokens: List[Tuple[str, int]] = []
    pos = 0
    while pos < len(text):
        matched = False
        for length in range(min(6, len(text) - pos), 0, -1):
            piece = text[pos:pos + length]
            if piece in vocab:
                kind = "unit" if piece in THAI_UNITS else "mult"
                tokens.append((kind, vocab[piece]))
                pos += length
                matched = True
                break
        if not matched:
            pos += 1

    if not tokens:
        return -1

    result = 0
    current = 0
    for kind, value in tokens:
        if kind == "unit":
            current = value
        else:
            if current == 0:
                current = 1
            result += current * value
            current = 0
    return result + current


def normalize_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()


def normalize_digits(text: str) -> str:
    return str(text or "").translate(THAI_DIGIT_MAP)


def extract_digits_only(text: str) -> str:
    text = str(text or "").strip()
    if not text:
        return ""

    thai_num = thai_text_to_number(text)
    if thai_num >= 0 and re.fullmatch(r"[\u0E00-\u0E7F\s]+", text):
        return str(thai_num)

    text = normalize_digits(text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = text.replace("|", " ")
    text = re.sub(r"[, ]", "", text)
    text = re.sub(r"[^0-9]", "", text)
    return text


def normalize_party_name(text: str) -> str:
    text = normalize_digits(str(text or ""))
    for token in [
        "\u0e1e\u0e23\u0e23\u0e04\u0e01\u0e32\u0e23\u0e40\u0e21\u0e37\u0e2d\u0e07",
        "\u0e0a\u0e37\u0e48\u0e2d\u0e1e\u0e23\u0e23\u0e04",
        "\u0e0a\u0e37\u0e48\u0e2d \u0e1e\u0e23\u0e23\u0e04",
    ]:
        text = text.replace(token, "")
    text = re.sub(r"\(.*?\)", " ", text)
    text = re.sub(r"[^\w\u0E00-\u0E7F]+", " ", text)
    return normalize_whitespace(text).lower()


def parse_doc_id_from_filename(filename: str) -> Tuple[str, int]:
    stem = Path(filename).stem
    match = re.match(r"^(.*?_\d+_\d+)(?:_page[_-]?(\d+))?$", stem)
    if not match:
        raise ValueError(f"Cannot parse filename: {filename}")
    return match.group(1), int(match.group(2) or 1)


def list_doc_pages(images_root: Path) -> Dict[str, List[Tuple[int, str]]]:
    grouped: Dict[str, List[Tuple[int, str]]] = defaultdict(list)
    for path in images_root.rglob("*"):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        rel = path.relative_to(images_root)
        if len(rel.parts) >= 2:
            doc_id = rel.parts[0]
            match = re.search(r"page[_-]?(\d+)", path.stem)
            page_num = int(match.group(1)) if match else 1
        else:
            doc_id, page_num = parse_doc_id_from_filename(path.name)
        grouped[doc_id].append((page_num, str(path)))
    return {doc_id: sorted(pages, key=lambda item: item[0]) for doc_id, pages in grouped.items()}


def resize_image(image: Image.Image) -> Image.Image:
    width, height = image.size
    scale = min(MAX_IMAGE_SIDE / max(width, height), 1.0)
    if scale < 1.0:
        return image.resize((int(width * scale), int(height * scale)))
    return image


def pil_to_jpeg_bytes(image: Image.Image) -> bytes:
    if image.mode != "RGB":
        image = image.convert("RGB")
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=JPEG_QUALITY, optimize=True)
    return buffer.getvalue()


def build_image_parts(image_pages: List[Tuple[int, str]]) -> List[Any]:
    parts = []
    for _, image_path in image_pages:
        image = Image.open(image_path)
        image = resize_image(image)
        parts.append(types.Part.from_bytes(data=pil_to_jpeg_bytes(image), mime_type="image/jpeg"))
    return parts


def is_rate_limit_error(error: Exception) -> bool:
    message = str(error)
    kind = type(error).__name__
    return "429" in message or "RESOURCE_EXHAUSTED" in message or "ResourceExhausted" in kind


def retry_generate(model: str, contents: List[Any], config: types.GenerateContentConfig):
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except Exception as error:
            last_error = error
            wait_seconds = RATE_LIMIT_WAIT if is_rate_limit_error(error) else BACKOFF_BASE * (2 ** attempt)
            print(f"[Retry {attempt + 1}/{MAX_RETRIES}] {type(error).__name__}: {error}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait_seconds)
    raise last_error


In [5]:
def extract_json_block(text: str) -> str:
    text = str(text or "").strip()
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]
    return text


def safe_json_loads(text: str) -> Dict[str, Any]:
    text = extract_json_block(text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", text)
    text = re.sub(r"\bTrue\b", "true", text)
    text = re.sub(r"\bFalse\b", "false", text)
    text = re.sub(r"\bNone\b", "null", text)
    text = re.sub(r",\s*([}\]])", r"\1", text)
    return json.loads(text)


def clean_rows(rows: List[Dict[str, Any]], doc_type: str, default_page: int = 1) -> List[Dict[str, Any]]:
    cleaned = []
    for index, row in enumerate(rows, start=1):
        if not isinstance(row, dict):
            continue
        item = dict(row)
        item["page"] = int(item.get("page", default_page) or default_page)
        item["row_order_on_page"] = int(item.get("row_order_on_page", index) or index)
        item["party_name"] = normalize_whitespace(item.get("party_name", ""))
        item["votes"] = extract_digits_only(item.get("votes", ""))
        if doc_type == "constituency":
            item["candidate_name"] = normalize_whitespace(item.get("candidate_name", ""))
        cleaned.append(item)
    cleaned.sort(key=lambda item: (int(item.get("page", 0)), int(item.get("row_order_on_page", 0))))
    return cleaned


def compute_row_sum(rows: List[Dict[str, Any]]) -> int:
    total = 0
    for row in rows:
        try:
            total += int(row.get("votes", "") or 0)
        except ValueError:
            pass
    return total


def validate_total_votes(rows: List[Dict[str, Any]], total_votes: str, allow_soft: bool = False) -> bool:
    if not total_votes:
        return True
    try:
        expected = int(total_votes)
    except ValueError:
        return True
    actual = compute_row_sum(rows)
    diff = abs(actual - expected)
    if diff == 0:
        return True
    if allow_soft and diff <= TOTAL_SOFT_TOLERANCE_ABS:
        return True
    print(f"  [TOTAL MISMATCH] sum={actual}, expected={expected}, diff={actual - expected}")
    return False


def rows_by_page(rows: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    grouped: Dict[int, List[Dict[str, Any]]] = defaultdict(list)
    for row in rows:
        grouped[int(row.get("page", 1))].append(row)
    return {
        page_num: sorted(items, key=lambda item: int(item.get("row_order_on_page", 0)))
        for page_num, items in grouped.items()
    }


def rebuild_all_rows(page_rows_map: Dict[int, List[Dict[str, Any]]]) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for page_num in sorted(page_rows_map):
        rows.extend(sorted(page_rows_map[page_num], key=lambda item: int(item.get("row_order_on_page", 0))))
    return rows


def page_priority_order(page_rows_map: Dict[int, List[Dict[str, Any]]], image_pages: List[Tuple[int, str]]) -> List[int]:
    last_page_num = max(page for page, _ in image_pages)
    scores = []
    for page_num, rows in page_rows_map.items():
        blank_votes = sum(1 for row in rows if not row.get("votes"))
        short_votes = sum(1 for row in rows if row.get("votes") and len(row.get("votes")) <= 2)
        score = blank_votes * 10 + short_votes * 2 + (5 if page_num == last_page_num else 0)
        scores.append((score, page_num))
    scores.sort(reverse=True)
    return [page_num for _, page_num in scores] or [page for page, _ in image_pages]


def page_debug_stats(page_rows_map: Dict[int, List[Dict[str, Any]]]) -> Dict[int, Dict[str, int]]:
    return {
        page_num: {
            "rows": len(rows),
            "blank_votes": sum(1 for row in rows if not row.get("votes")),
            "sum": compute_row_sum(rows),
        }
        for page_num, rows in sorted(page_rows_map.items())
    }


## Template and Crop Metadata


In [6]:
def load_template_dataframe(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    party_col = next((col for col in df.columns if str(col).startswith("party_name")), None)
    if party_col is None:
        raise ValueError("Could not find a party_name column in the template.")
    df = df.rename(columns={party_col: "party_name"}).copy()
    if "doc_id" not in df.columns:
        df["doc_id"] = df["id"].str.rsplit("_", n=1).str[0]
    if "row_num" not in df.columns:
        df["row_num"] = df["id"].str.rsplit("_", n=1).str[-1].astype(int)
    df["party_name"] = df["party_name"].fillna("").astype(str)
    df["party_name_norm"] = df["party_name"].map(normalize_party_name)
    return df


template_df = load_template_dataframe(TEMPLATE_PATH)
doc_to_template_rows = {
    doc_id: group.sort_values("row_num").copy()
    for doc_id, group in template_df.groupby("doc_id", sort=False)
}


def normalize_slashes(path_like: str) -> str:
    return str(path_like or "").replace("\\", "/")


def resolve_preprocessed_path(path_like: str, artifact_folder: str) -> Path | None:
    if PREPROCESSED_ROOT is None:
        return None
    raw = normalize_slashes(path_like)
    candidates: List[Path] = []
    for marker in [f"/preprocessed/{artifact_folder}/", f"/{artifact_folder}/"]:
        if marker in raw:
            suffix = raw.split(marker, 1)[1]
            candidates.append(PREPROCESSED_ROOT / artifact_folder / suffix)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    name = Path(raw).name
    if name:
        matches = list((PREPROCESSED_ROOT / artifact_folder).rglob(name))
        if matches:
            return matches[0]
    return None


def load_crop_lookup(rows_manifest_path: Path | None) -> Dict[str, Dict[Tuple[int, int], Dict[str, Path | None]]]:
    if not rows_manifest_path or not rows_manifest_path.exists():
        return {}

    rows_df = pd.read_csv(rows_manifest_path)
    lookup: Dict[str, Dict[Tuple[int, int], Dict[str, Path | None]]] = defaultdict(dict)
    for row in rows_df.itertuples(index=False):
        row_path = resolve_preprocessed_path(getattr(row, "row_path", ""), "rows")
        vote_cell_path = resolve_preprocessed_path(getattr(row, "vote_cell_path", ""), "vote_cells")
        key = (int(getattr(row, "page_num")), int(getattr(row, "row_index_in_page")))
        lookup[getattr(row, "doc_id")][key] = {
            "row_path": row_path,
            "vote_cell_path": vote_cell_path,
        }
    return {doc_id: dict(page_map) for doc_id, page_map in lookup.items()}


CROP_LOOKUP = load_crop_lookup(ROWS_MANIFEST_PATH)

print(f"Template rows: {len(template_df)}")
print(f"Docs in template: {template_df['doc_id'].nunique()}")
print(f"Docs with crop metadata: {len(CROP_LOOKUP)}")


Template rows: 10053
Docs in template: 300
Docs with crop metadata: 280


## Prompts and Gemini Calls


In [7]:
SYSTEM_PROMPT = """
You extract structured Thai election vote-table data from page images.

Rules:
1. Return only valid JSON.
2. Keep rows in visual order from top to bottom.
3. Output votes with digits only.
4. Convert Thai numerals to Arabic digits.
5. Ignore headers, seals, signatures, and handwritten marks outside the table.
6. Only output rows that are clearly visible.
7. Only output total_votes when the page clearly contains the grand total row.
"""

TOTAL_ROW_HINTS = [
    "\\u0e23\\u0e27\\u0e21\\u0e04\\u0e30\\u0e41\\u0e19\\u0e19\\u0e17\\u0e31\\u0e49\\u0e07\\u0e2a\\u0e34\\u0e49\\u0e19",
    "\\u0e23\\u0e27\\u0e21\\u0e04\\u0e30\\u0e41\\u0e19\\u0e19\\u0e17\\u0e31\\u0e49\\u0e07\\u0e2b\\u0e21\\u0e14",
    "\\u0e23\\u0e27\\u0e21\\u0e04\\u0e30\\u0e41\\u0e19\\u0e19",
]


def call_gemini_json(parts: List[Any], save_prefix: str, temperature: float = 0.0) -> Dict[str, Any]:
    config = types.GenerateContentConfig(
        temperature=temperature,
        top_p=0.95,
        top_k=32,
        max_output_tokens=16384,
        response_mime_type="application/json",
    )
    contents = [types.Content(role="user", parts=parts)]
    response = retry_generate(MODEL_NAME, contents, config)
    raw_text = response.text or ""
    try:
        return safe_json_loads(raw_text)
    except Exception:
        if SAVE_FAILED_RAW_ONLY:
            (FAILED_RAW_DIR / f"{save_prefix}.txt").write_text(raw_text, encoding="utf-8")
        repair_parts = [types.Part.from_text(text="Return only valid JSON."), *parts]
        repair_response = retry_generate(MODEL_NAME, [types.Content(role="user", parts=repair_parts)], config)
        repaired_text = repair_response.text or ""
        try:
            return safe_json_loads(repaired_text)
        except Exception:
            if SAVE_FAILED_RAW_ONLY:
                (FAILED_RAW_DIR / f"{save_prefix}_retry.txt").write_text(repaired_text, encoding="utf-8")
            raise


def build_single_page_prompt(doc_id: str, doc_type: str, page_num: int) -> str:
    row_schema = (
        '{"page": 1, "row_order_on_page": 1, "party_name": "...", "votes": "12345"}'
        if doc_type == "party_list"
        else '{"page": 1, "row_order_on_page": 1, "candidate_name": "...", "party_name": "...", "votes": "12345"}'
    )
    return f"""
Document ID: {doc_id}
Document type: {doc_type}
Page: {page_num}

Extract all visible vote-table rows on this page.
Only set has_grand_total_row=true if you can clearly see the final summary row.
Grand total hints: {TOTAL_ROW_HINTS}

Return JSON like:
{{
  "doc_id": "{doc_id}",
  "doc_type": "{doc_type}",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [{row_schema}]
}}
"""


def build_doc_rescue_prompt(
    doc_id: str,
    doc_type: str,
    page_nums: List[int],
    expected_total: int,
    previous_rows: List[Dict[str, Any]],
    template_count: int,
) -> str:
    preview = json.dumps(
        [
            {
                "page": row.get("page"),
                "row_order_on_page": row.get("row_order_on_page"),
                "party_name": row.get("party_name", ""),
                "candidate_name": row.get("candidate_name", ""),
                "votes": row.get("votes", ""),
            }
            for row in previous_rows[: min(len(previous_rows), 120)]
        ],
        ensure_ascii=False,
        indent=2,
    )
    row_schema = (
        '{"page": 1, "row_order_on_page": 1, "party_name": "...", "votes": "12345"}'
        if doc_type == "party_list"
        else '{"page": 1, "row_order_on_page": 1, "candidate_name": "...", "party_name": "...", "votes": "12345"}'
    )
    return f"""
Document ID: {doc_id}
Document type: {doc_type}
Pages provided: {page_nums}

Read all pages together as one document.
The official grand total should be {expected_total}.
The document should contain about {template_count} rows.
Re-check every digit carefully.
Return rows in page order, then top-to-bottom order.

Previous extraction:
{preview}

Return JSON like:
{{
  "doc_id": "{doc_id}",
  "doc_type": "{doc_type}",
  "has_grand_total_row": true,
  "total_votes": "{expected_total}",
  "rows": [{row_schema}]
}}
"""


In [8]:
def build_recheck_prompt(
    doc_id: str,
    doc_type: str,
    page_num: int,
    expected_total: int,
    current_sum: int,
    previous_rows: List[Dict[str, Any]],
) -> str:
    preview = json.dumps(
        [
            {
                "row_order_on_page": row.get("row_order_on_page"),
                "party_name": row.get("party_name", ""),
                "candidate_name": row.get("candidate_name", ""),
                "votes": row.get("votes", ""),
            }
            for row in previous_rows
        ],
        ensure_ascii=False,
        indent=2,
    )
    row_schema = (
        '{"page": 1, "row_order_on_page": 1, "party_name": "...", "votes": "12345"}'
        if doc_type == "party_list"
        else '{"page": 1, "row_order_on_page": 1, "candidate_name": "...", "party_name": "...", "votes": "12345"}'
    )
    return f"""
Document ID: {doc_id}
Document type: {doc_type}
Page: {page_num}
Official grand total on the full document: {expected_total}
Current extracted document sum: {current_sum}

Re-read only this page.
Focus on vote digits that might be wrong or missing.
Keep rows in visual order from top to bottom.

Previous extraction:
{preview}

Return JSON like:
{{
  "doc_id": "{doc_id}",
  "doc_type": "{doc_type}",
  "page": {page_num},
  "has_grand_total_row": false,
  "total_votes": "",
  "rows": [{row_schema}]
}}
"""


def build_crop_prompt(doc_id: str, page_num: int, row_order_on_page: int) -> str:
    return f"""
Read only the vote number from this Thai election crop.
Document ID: {doc_id}
Page: {page_num}
Row on page: {row_order_on_page}

Return JSON exactly like:
{{
  "votes": "12345"
}}

If unreadable, return:
{{
  "votes": ""
}}
"""


def extract_single_page_with_gemini(doc_id: str, page_num: int, image_path: str) -> Tuple[List[Dict[str, Any]], str, bool]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(text=build_single_page_prompt(doc_id, doc_type, page_num)),
        *build_image_parts([(page_num, image_path)]),
    ]
    data = call_gemini_json(parts, save_prefix=f"{doc_id}_page{page_num}", temperature=0.0)
    rows = clean_rows(data.get("rows", []), doc_type, default_page=page_num)
    has_total = bool(data.get("has_grand_total_row", False))
    total_votes = extract_digits_only(data.get("total_votes", "")) if has_total else ""
    return rows, total_votes, has_total


def extract_multi_page_rescue(
    doc_id: str,
    image_pages: List[Tuple[int, str]],
    expected_total: int,
    previous_rows: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], str]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    page_nums = [page_num for page_num, _ in image_pages]
    template_count = len(doc_to_template_rows.get(doc_id, []))
    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(
            text=build_doc_rescue_prompt(doc_id, doc_type, page_nums, expected_total, previous_rows, template_count)
        ),
        *build_image_parts(image_pages),
    ]
    data = call_gemini_json(parts, save_prefix=f"{doc_id}_doc_rescue", temperature=0.1)
    rows = clean_rows(data.get("rows", []), doc_type, default_page=page_nums[0] if page_nums else 1)
    has_total = bool(data.get("has_grand_total_row", False))
    total_votes = extract_digits_only(data.get("total_votes", "")) if has_total else str(expected_total)
    return rows, total_votes


def targeted_recheck_page(
    doc_id: str,
    page_num: int,
    image_path: str,
    previous_rows: List[Dict[str, Any]],
    expected_total: int,
    current_sum: int,
) -> Tuple[List[Dict[str, Any]], str]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    parts = [
        types.Part.from_text(text=SYSTEM_PROMPT),
        types.Part.from_text(
            text=build_recheck_prompt(doc_id, doc_type, page_num, expected_total, current_sum, previous_rows)
        ),
        *build_image_parts([(page_num, image_path)]),
    ]
    data = call_gemini_json(parts, save_prefix=f"{doc_id}_page{page_num}_recheck", temperature=0.1)
    rows = clean_rows(data.get("rows", []), doc_type, default_page=page_num)
    has_total = bool(data.get("has_grand_total_row", False))
    total_votes = extract_digits_only(data.get("total_votes", "")) if has_total else ""
    return rows or previous_rows, total_votes


def read_vote_from_crop(doc_id: str, page_num: int, row_order_on_page: int, image_path: Path) -> str:
    parts = [
        types.Part.from_text(text=build_crop_prompt(doc_id, page_num, row_order_on_page)),
        *build_image_parts([(page_num, str(image_path))]),
    ]
    try:
        data = call_gemini_json(parts, save_prefix=f"{doc_id}_p{page_num}_r{row_order_on_page}_crop", temperature=0.0)
        return extract_digits_only(data.get("votes", ""))
    except Exception as error:
        print(f"  [CROP ERROR] {doc_id} page {page_num} row {row_order_on_page}: {type(error).__name__}: {error}")
        return ""


def rescue_rows_with_crops(doc_id: str, rows: List[Dict[str, Any]], allow_short_votes: bool = False) -> List[Dict[str, Any]]:
    if not ENABLE_CROP_RESCUE or doc_id not in CROP_LOOKUP:
        return rows

    rescued = []
    for row in rows:
        current_vote = row.get("votes", "")
        needs_rescue = current_vote == "" or (allow_short_votes and len(current_vote) <= 2)
        if not needs_rescue:
            rescued.append(row)
            continue

        key = (int(row.get("page", 1)), int(row.get("row_order_on_page", 1)))
        crop_info = CROP_LOOKUP[doc_id].get(key)
        if not crop_info:
            rescued.append(row)
            continue

        rescued_vote = ""
        for image_key in ("vote_cell_path", "row_path"):
            image_path = crop_info.get(image_key)
            if image_path and image_path.exists():
                rescued_vote = read_vote_from_crop(doc_id, key[0], key[1], image_path)
                if rescued_vote:
                    break

        if rescued_vote:
            updated = dict(row)
            updated["votes"] = rescued_vote
            rescued.append(updated)
        else:
            rescued.append(row)
    return rescued


## Matching and Document Pipeline


In [9]:
def best_match_row(extracted_party: str, template_party_norms: List[str]) -> Tuple[int, int]:
    party_norm = normalize_party_name(extracted_party)
    if not party_norm:
        return -1, 0
    best = process.extractOne(party_norm, template_party_norms, scorer=fuzz.WRatio)
    if best is None:
        return -1, 0
    _, score, index = best
    return index, int(score)


def map_party_list_by_order_if_possible(doc_id: str, extracted_rows: List[Dict[str, Any]]) -> Dict[int, str]:
    template_rows = doc_to_template_rows[doc_id].copy()
    row_nums = template_rows["row_num"].tolist()
    ordered_rows = sorted(extracted_rows, key=lambda row: (int(row.get("page", 0)), int(row.get("row_order_on_page", 0))))
    if len(ordered_rows) != len(row_nums):
        return {}
    mapping = {}
    for row_num, row in zip(row_nums, ordered_rows):
        vote = extract_digits_only(row.get("votes", ""))
        if vote == "":
            return {}
        mapping[row_num] = vote
    return mapping


def map_extracted_rows_to_template(doc_id: str, extracted_doc: Dict[str, Any]) -> Dict[int, str]:
    template_rows = doc_to_template_rows[doc_id].copy()
    template_party_norms = template_rows["party_name_norm"].tolist()
    row_nums = template_rows["row_num"].tolist()
    candidates = extracted_doc.get("rows", [])

    if doc_id.startswith("party_list_"):
        ordered_mapping = map_party_list_by_order_if_possible(doc_id, candidates)
        if ordered_mapping:
            return ordered_mapping

    mapping: Dict[int, str] = {}
    used_indices = set()
    scored_matches = []

    for row in candidates:
        vote = extract_digits_only(row.get("votes", ""))
        party_name = row.get("party_name", "")
        if vote == "" or not party_name:
            continue
        index, score = best_match_row(party_name, template_party_norms)
        if index >= 0:
            scored_matches.append((score, index, row))

    scored_matches.sort(reverse=True, key=lambda item: item[0])

    for score, index, row in scored_matches:
        if index in used_indices or score < STRICT_MATCH_THRESHOLD:
            continue
        used_indices.add(index)
        mapping[row_nums[index]] = extract_digits_only(row.get("votes", ""))

    remaining_indices = [index for index in range(len(template_party_norms)) if index not in used_indices]
    for row in candidates:
        vote = extract_digits_only(row.get("votes", ""))
        party_name = row.get("party_name", "")
        if vote == "" or not party_name:
            continue

        party_norm = normalize_party_name(party_name)
        best_index = -1
        best_score = -1
        for index in remaining_indices:
            score = fuzz.WRatio(party_norm, template_party_norms[index])
            if score > best_score:
                best_score = score
                best_index = index

        if best_index >= 0 and best_score >= LOOSE_MATCH_THRESHOLD:
            row_num = row_nums[best_index]
            if row_num not in mapping:
                mapping[row_num] = vote
                used_indices.add(best_index)
                remaining_indices.remove(best_index)

    return mapping


def extract_doc_rows_with_gemini(doc_id: str, image_pages: List[Tuple[int, str]]) -> Dict[str, Any]:
    doc_type = "party_list" if doc_id.startswith("party_list_") else "constituency"
    page_rows_map: Dict[int, List[Dict[str, Any]]] = {}
    doc_total_votes = ""
    last_page_num = max(page_num for page_num, _ in image_pages)

    for page_num, image_path in image_pages:
        try:
            rows, page_total, has_total = extract_single_page_with_gemini(doc_id, page_num, image_path)
            page_rows_map[page_num] = rows
            if has_total and page_num == last_page_num and page_total:
                doc_total_votes = page_total
            accepted = page_total if has_total and page_num == last_page_num and page_total else ""
            print(
                f"  page {page_num}: extracted {len(rows)} rows"
                + (f", raw_total={page_total}, accepted_total={accepted}" if page_total else "")
            )
        except Exception as error:
            print(f"  [PAGE ERROR] {doc_id} page {page_num}: {type(error).__name__}: {error}")
            page_rows_map[page_num] = []

    all_rows = rebuild_all_rows(page_rows_map)
    all_rows = rescue_rows_with_crops(doc_id, all_rows, allow_short_votes=False)
    page_rows_map = rows_by_page(all_rows)
    template_count = len(doc_to_template_rows.get(doc_id, []))
    blank_votes = sum(1 for row in all_rows if not row.get("votes"))
    skip_total_repair = len(image_pages) == 1 and len(all_rows) == template_count and blank_votes == 0

    if doc_total_votes:
        print(f"  page sums: {page_debug_stats(page_rows_map)}")
    if skip_total_repair:
        print("  [FAST PATH] skip total-repair for complete single-page doc")

    if doc_total_votes and not skip_total_repair and not validate_total_votes(all_rows, doc_total_votes):
        expected_total = int(doc_total_votes)
        current_diff = abs(compute_row_sum(all_rows) - expected_total)
        reference = max(compute_row_sum(all_rows), expected_total, 1)

        if current_diff > reference * TOTAL_HARD_SKIP_PCT:
            print(f"  [TOTAL SKIP] diff={current_diff} > {int(reference * TOTAL_HARD_SKIP_PCT)}")
        else:
            try:
                rescue_rows, rescue_total = extract_multi_page_rescue(doc_id, image_pages, expected_total, all_rows)
                rescue_rows = rescue_rows_with_crops(doc_id, rescue_rows, allow_short_votes=False)
                rescue_diff = abs(compute_row_sum(rescue_rows) - expected_total)
                print(f"  [DOC RESCUE] diff {current_diff} -> {rescue_diff}")
                if rescue_rows and rescue_diff < current_diff:
                    all_rows = rescue_rows
                    page_rows_map = rows_by_page(all_rows)
                    current_diff = rescue_diff
                    if rescue_total:
                        doc_total_votes = rescue_total
            except Exception as error:
                print(f"  [DOC RESCUE ERROR] {doc_id}: {type(error).__name__}: {error}")

            page_path_map = {page_num: path for page_num, path in image_pages}
            for round_index in range(1, TOTAL_VALIDATION_ROUNDS + 1):
                if validate_total_votes(all_rows, doc_total_votes, allow_soft=True):
                    break

                improved = False
                order = page_priority_order(page_rows_map, image_pages)
                print(f"  [TOTAL REPAIR ROUND {round_index}/{TOTAL_VALIDATION_ROUNDS}] page order={order}")

                for page_num in order:
                    current_rows = page_rows_map.get(page_num, [])
                    new_rows, page_total = targeted_recheck_page(
                        doc_id=doc_id,
                        page_num=page_num,
                        image_path=page_path_map[page_num],
                        previous_rows=current_rows,
                        expected_total=expected_total,
                        current_sum=compute_row_sum(all_rows),
                    )
                    new_rows = rescue_rows_with_crops(
                        doc_id,
                        [row for row in new_rows if int(row.get("page", page_num)) == page_num],
                        allow_short_votes=False,
                    )

                    candidate_map = dict(page_rows_map)
                    candidate_map[page_num] = new_rows
                    candidate_rows = rebuild_all_rows(candidate_map)
                    candidate_diff = abs(compute_row_sum(candidate_rows) - expected_total)

                    if candidate_diff < current_diff:
                        print(f"    [ACCEPT PAGE {page_num}] diff {current_diff} -> {candidate_diff}")
                        page_rows_map = candidate_map
                        all_rows = candidate_rows
                        current_diff = candidate_diff
                        improved = True
                        if page_total and page_num == last_page_num:
                            doc_total_votes = page_total
                        if validate_total_votes(all_rows, doc_total_votes, allow_soft=True):
                            break
                    else:
                        print(f"    [REJECT PAGE {page_num}] diff would be {current_diff} -> {candidate_diff}")

                if not improved:
                    print("  [STOP] no page improved global diff")
                    break

    return {
        "doc_id": doc_id,
        "doc_type": doc_type,
        "total_votes": doc_total_votes,
        "rows": all_rows,
    }


## Run and Build Submission


In [16]:
doc_cache = load_json(DOC_CACHE_PATH, {})
row_cache = load_json(ROW_CACHE_PATH, {})
progress = load_json(PROGRESS_PATH, {"done_docs": [], "errors": {}})

done_docs = set(progress.get("done_docs", []))
errors = progress.get("errors", {})

doc_pages = list_doc_pages(IMAGES_ROOT)
all_doc_ids = sorted(doc_pages)

print(f"Total docs found: {len(all_doc_ids)}")
print(f"Already done    : {len(done_docs)}")

for doc_id in tqdm(all_doc_ids):
    if doc_id in done_docs:
        continue
    if doc_id not in doc_to_template_rows:
        # print(f"[SKIP] {doc_id} not found in template")
        continue

    try:
        pages = doc_pages[doc_id]
        print("\n" + "=" * 90)
        # print(f"[DOC] {doc_id}")
        print("pages:", [page_num for page_num, _ in pages])

        extracted_doc = extract_doc_rows_with_gemini(doc_id, pages)
        doc_cache[doc_id] = extracted_doc

        row_map = map_extracted_rows_to_template(doc_id, extracted_doc)
        row_cache[doc_id] = {str(row_num): vote for row_num, vote in row_map.items()}

        # print(f"Extracted rows: {len(extracted_doc.get('rows', []))}")
        # print(f"Matched rows  : {len(row_map)} / {len(doc_to_template_rows[doc_id])}")

        done_docs.add(doc_id)
        errors.pop(doc_id, None)
        save_json(DOC_CACHE_PATH, doc_cache)
        save_json(ROW_CACHE_PATH, row_cache)
        save_json(PROGRESS_PATH, {"done_docs": sorted(done_docs), "errors": errors})
    except Exception as error:
        errors[doc_id] = {
            "error_type": type(error).__name__,
            "error": str(error),
            "traceback": traceback.format_exc()[-4000:],
        }
        save_json(PROGRESS_PATH, {"done_docs": sorted(done_docs), "errors": errors})
        # print(f"[ERROR] {doc_id}: {type(error).__name__}: {error}")

Total docs found: 280
Already done    : 90


  0%|          | 0/280 [00:00<?, ?it/s]

In [19]:
pred_votes = []
for row in template_df.itertuples(index=False):
    row_num = int(getattr(row, "row_num"))
    doc_id = getattr(row, "doc_id")
    vote = row_cache.get(doc_id, {}).get(str(row_num), "")
    pred_votes.append(extract_digits_only(vote))

submission_df = template_df[["id"]].copy()
submission_df["votes"] = pred_votes
if FILL_EMPTY_WITH_ZERO:
    submission_df["votes"] = submission_df["votes"].replace("", "0")

submission_df.to_csv(OUTPUT_PATH, index=False)

print("\n" + "=" * 90)
print(f"Saved submission : {OUTPUT_PATH}")
print(f"Total rows       : {len(submission_df)}")
print(f"Zero votes       : {(submission_df['votes'] == '0').sum()}")
print(f"Documents done   : {len(done_docs)}")
print(f"Errors           : {len(errors)}")
submission_df.head(20)


Saved submission : /kaggle/working/gemini_clean_plus/submission_gemini_clean_plus.csv
Total rows       : 10053
Zero votes       : 0
Documents done   : 280
Errors           : 0


,id,votes
0,constituency_10_1_1,14833
1,constituency_10_1_2,14368
2,constituency_10_1_3,979
3,constituency_10_1_4,244
4,constituency_10_1_5,351
5,constituency_10_1_6,34167
6,constituency_10_1_7,6030
7,constituency_10_1_8,1013
8,constituency_10_1_9,2075
9,constituency_10_1_10,168
